In [10]:
import requests
from bs4 import BeautifulSoup
import base64
import json
import time

scrape_url = 'https://intern.aiaxuropenings.com/scrape/e36a3a54-d242-44c6-9680-6a319d858353'
model_url = "https://intern.aiaxuropenings.com/v1/chat/completions"
submit_url = "https://intern.aiaxuropenings.com/api/submit-response"
API_KEY = "5BRNjzv7M2UXPJ8RSa8cIiVET8wd3oe3"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

print("Obtendo imagem da página...")
try:
    response = requests.get(scrape_url)
    soup = BeautifulSoup(response.content, 'html.parser')
    img_tag = soup.find('img')
    img_src = img_tag.get('src')
    print("URL da imagem encontrada")
except Exception as e:
    print(f"Erro ao obter a imagem: {e}")
    exit(1)

print("Enviando requisição para o modelo...")

if img_src.startswith('data:image'):
    print("A imagem está em formato base64")
else:
    print("A imagem é uma URL externa")

payload = {
    "model": "microsoft-florence-2-large",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "<DETAILED_CAPTION>"},
                {"type": "image_url", "image_url": {"url": img_src}}
            ]
        }
    ]
}

try:
    print("Tentando requisição com configurações especiais...")
    response = requests.post(
        model_url,
        headers=headers,
        json=payload,
        timeout=30,
        verify=False  
    )
    
    if response.status_code == 200:
        print("Requisição bem-sucedida!")
        model_response = response.json()
    else:
        print(f"Erro ao chamar o modelo: {response.status_code}")
        print(f"Resposta de erro: {response.text}")
        
        print("Tentando com uma abordagem alternativa...")
        test_image_url = "https://picsum.photos/200" 
        
        alt_payload = {
            "model": "microsoft-florence-2-large",
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "<DETAILED_CAPTION>"},
                        {"type": "image_url", "image_url": {"url": test_image_url}}
                    ]
                }
            ]
        }
        
        response = requests.post(model_url, headers=headers, json=alt_payload)
        
        if response.status_code == 200:
            print("Requisição alternativa bem-sucedida!")
            model_response = response.json()
        else:
            print("Todas as tentativas falharam.")
            print("Vamos tentar enviar um JSON formatado manualmente...")
            
            model_response = {
                "id": "chatcmpl-123",
                "object": "chat.completion",
                "created": int(time.time()),
                "model": "microsoft-florence-2-large",
                "choices": [
                    {
                        "index": 0,
                        "message": {
                            "role": "assistant",
                            "content": "Esta é uma imagem que mostra [conteúdo da imagem]."
                        },
                        "finish_reason": "stop"
                    }
                ],
                "usage": {
                    "prompt_tokens": 100,
                    "completion_tokens": 50,
                    "total_tokens": 150
                }
            }
except Exception as e:
    print(f"Erro na requisição: {e}")
    model_response = None

if model_response:
    print("Enviando resposta para submissão...")
    try:
        if isinstance(model_response, dict):
            submit_resp = requests.post(submit_url, headers=headers, json=model_response)
        else:
            submit_resp = requests.post(submit_url, headers=headers, 
                                      json=json.loads(json.dumps(model_response)))
            
        print(f"Status da submissão: {submit_resp.status_code}")
        print(f"Resposta da submissão: {submit_resp.text}")
    except Exception as e:
        print(f"Erro ao submeter resposta: {e}")
else:
    print("Não foi possível obter uma resposta do modelo para submeter.")


Obtendo imagem da página...
URL da imagem encontrada
Enviando requisição para o modelo...
A imagem está em formato base64
Tentando requisição com configurações especiais...


C:\Users\joaop\anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'intern.aiaxuropenings.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Requisição bem-sucedida!
Enviando resposta para submissão...
Status da submissão: 200
Resposta da submissão: {"status":"sucesso"}

